# Day 2 · Partitions and shuffles

**Objective:** produce a runtime-against-shuffle-partition-count table. **That table is the answer
to "have you tuned a Spark job."** Everything else today supports it.

Reference: `knowledge_hub/skills/spark/02_partitions_and_shuffles.md`

---

### Carried over from Day 1

> 200 shuffle partitions for 262 zones on 2 cores. What is the right number, and how do I find it?

### ⚠️ Three different things share the word "partition"

Mixing these up is a visible tell in an interview.

| Term | Set by | Controls |
|---|---|---|
| **Input partitions** | file size, `spark.sql.files.maxPartitionBytes` | how the **read** is split. Yours was 3. |
| **Shuffle partitions** | `spark.sql.shuffle.partitions`, default **200** | how many partitions exist **after** a wide op |
| **`partitionBy` on write** | you | **directory layout on disk**. Unrelated to the other two. |

### ⚠️ Set expectations before you measure

Your dataset is 153 MB and `groupBy().count()` pre-aggregates on the map side, so only around 786
rows actually cross the shuffle. **The tuning table will mostly measure task scheduling overhead,
not data movement.** That is still exactly the lesson (200 tasks that each do nothing still cost
200 task setups on 2 cores), but say it that way rather than claiming you shrank a big shuffle.

If you want the numbers to bite harder, run `make data-full` for all twelve months first.

## Setup

In [1]:
# Plumbing, not Spark.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [3]:
# Open the session for today.
#   - import get_spark and YELLOW from src.session
#   - name it "day2"
#   - AQE stays off (the default in get_spark). It coalesces shuffle partitions
#     automatically, which is correct in production and would erase the entire
#     experiment you are about to run.
from src.session import get_spark, YELLOW
spark = get_spark("day2")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/14 12:07:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/14 12:08:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark 3.5.9 | AQE=False | UI http://wc-allauddin-shaik-shared-vpc:4041


In [4]:
# Read the parquet into df.
df = spark.read.parquet(YELLOW)

---
## E1 · Where input partitions come from

You measured 3 on Day 1. Now derive **why** 3, so the number stops being a fact you remember and
becomes one you can reproduce on any dataset.

In [14]:
# Print the input partition count, the file count, and the total size on disk.
#
# Then work the formula by hand:
#   maxSplitBytes = min(maxPartitionBytes, max(openCostInBytes, totalBytes / cores))
# where totalBytes counts (fileSize + openCostInBytes) for every file.
#
# Read the three config values off the session rather than trusting memory:
#   spark.sql.files.maxPartitionBytes
#   spark.sql.files.openCostInBytes
#   spark.sparkContext.defaultParallelism
print(f"Partition count: {df.rdd.getNumPartitions()}")

files = df.inputFiles()
print(f"File count: {len(files)}")

sc = spark.sparkContext
hconf = sc._jsc.hadoopConfiguration()
HPath = sc._jvm.org.apache.hadoop.fs.Path

p = HPath(str(Path(YELLOW).parent))
fs = p.getFileSystem(hconf)

total = 0
for st in fs.listStatus(p):
    if st.getPath().getName().endswith(".parquet"):
        mb = st.getLen() / 1024**2
        total += st.getLen()
        print(st.getPath().getName(), round(mb, 1), "MB")
print("total:", round(total / 1024**2, 1), "MB")


for k in ["spark.sql.files.maxPartitionBytes",
          "spark.sql.files.openCostInBytes",
          "spark.sql.shuffle.partitions"]:
    print(k, "=", spark.conf.get(k))

Partition count: 3
File count: 3
yellow_tripdata_2024-02.parquet 48.0 MB
yellow_tripdata_2024-01.parquet 47.6 MB
yellow_tripdata_2024-03.parquet 57.3 MB
total: 153.0 MB
spark.sql.files.maxPartitionBytes = 134217728b
spark.sql.files.openCostInBytes = 4194304b
spark.sql.shuffle.partitions = 200


In [16]:
import os
os.cpu_count()

2

**Fill this in:**

| | |
|---|---|
| Input partitions | 3|
| Files | 3|
| `maxPartitionBytes` | 128 mb|
| `openCostInBytes` | 4 mb|
| Cores | 2 |
| `maxSplitBytes` I calculated | 51|
| Was the binding constraint 128 MB or `totalBytes / cores`? |`totalBytes / cores`|

---
## E2 · ⭐ The tuning table

**This is the deliverable of the whole day.** Everything before it is setup and everything after it
is supporting evidence.

`spark.sql.shuffle.partitions` defaults to **200 regardless of data size**, which is the single most
common Spark misconfiguration. On a few GB you get tiny partitions and pay task setup 200 times. On
a few TB you get partitions that do not fit in memory and spill.

Rough working target: **100 MB to 200 MB per shuffle partition**, and never fewer than your core count.

In [ ]:
# Write a small timing helper. Something that takes a zero-argument function,
# runs it, and returns elapsed seconds rounded.
#
# Do not use %%time here: you need the number as a value so you can collect it
# into a table, not printed text.

In [ ]:
# The experiment. For each n in [200, 100, 64, 32, 16, 8, 4, 2]:
#   - set spark.sql.shuffle.partitions to n
#   - time the same aggregation each round (groupBy PULocationID, count, collect)
#   - record n and the seconds
#
# Two things that will ruin the numbers if you skip them:
#   - run the aggregation once before the loop and throw the result away. The first
#     run pays JVM warm-up and file-handle costs that belong to no particular n.
#   - keep the query identical across rounds. Only n changes.

In [17]:
import time
from pyspark.sql import functions as F

sc = spark.sparkContext


def q1():
    """LOW cardinality: 262 groups. Map-side combine collapses ~9.5M rows to ~786
    before the shuffle, so almost nothing moves however many partitions we ask for."""
    (df.groupBy("PULocationID")
       .agg(F.count("*").alias("trips"))
       .write.mode("overwrite").format("noop").save())


def q2():
    """HIGH cardinality: zone x zone x calendar hour, millions of groups.
    Map-side combine can barely collapse anything, so real rows cross the shuffle."""
    (df.groupBy("PULocationID",
                "DOLocationID",
                F.date_trunc("hour", "tpep_pickup_datetime").alias("pickup_hour"))
       .agg(F.count("*").alias("trips"),
            F.avg("total_amount").alias("avg_fare"),
            F.sum("trip_distance").alias("total_miles"))
       .write.mode("overwrite").format("noop").save())


def tune(query, name, ns=(200, 100, 64, 32, 16, 8, 4, 2)):
    """Time `query` once per value of spark.sql.shuffle.partitions."""
    # warm-up, thrown away: the JVM compiles hot paths on the first run
    spark.conf.set("spark.sql.shuffle.partitions", 200)
    sc.setJobDescription(f"{name} WARMUP")
    query()

    rows = []
    for n in ns:
        spark.conf.set("spark.sql.shuffle.partitions", n)
        sc.setJobDescription(f"{name} n={n}")        # labels the job in the UI
        t = time.perf_counter()
        query()
        secs = round(time.perf_counter() - t, 2)
        rows.append((n, secs))
        print(f"{name}  n={n:>4}  {secs:>7.2f}s")

    sc.setJobDescription(None)
    return rows


def as_markdown(rows, title):
    print(f"\n**{title}**\n")
    print("| n | seconds |")
    print("|---|---|")
    for n, s in rows:
        print(f"| {n} | {s} |")

In [18]:
r1 = tune(q1, "q1-low-card")
as_markdown(r1, "Query 1 · low cardinality")

q1-low-card  n= 200     5.95s


q1-low-card  n= 100     2.27s


q1-low-card  n=  64     1.90s


q1-low-card  n=  32     1.65s


q1-low-card  n=  16     2.19s
q1-low-card  n=   8     0.91s
q1-low-card  n=   4     0.87s
q1-low-card  n=   2     0.82s

**Query 1 · low cardinality**

| n | seconds |
|---|---|
| 200 | 5.95 |
| 100 | 2.27 |
| 64 | 1.9 |
| 32 | 1.65 |
| 16 | 2.19 |
| 8 | 0.91 |
| 4 | 0.87 |
| 2 | 0.82 |


In [19]:
r2 = tune(q2, "q2-high-card")
as_markdown(r2, "Query 2 · high cardinality")

q2-high-card  n= 200    14.66s


q2-high-card  n= 100    19.35s


q2-high-card  n=  64    17.46s


q2-high-card  n=  32    11.34s


q2-high-card  n=  16    10.76s


q2-high-card  n=   8    11.11s


q2-high-card  n=   4    10.25s


q2-high-card  n=   2    10.26s

**Query 2 · high cardinality**

| n | seconds |
|---|---|
| 200 | 14.66 |
| 100 | 19.35 |
| 64 | 17.46 |
| 32 | 11.34 |
| 16 | 10.76 |
| 8 | 11.11 |
| 4 | 10.25 |
| 2 | 10.26 |


In [20]:
spark.conf.set("spark.sql.adaptive.enabled", True)
spark.conf.set("spark.sql.shuffle.partitions", 200)
sc.setJobDescription("q2 AQE ON")

t = time.perf_counter()
q2()
print("AQE on, starting from 200:", round(time.perf_counter() - t, 2), "s")

spark.conf.set("spark.sql.adaptive.enabled", False)
sc.setJobDescription(None)

AQE on, starting from 200: 13.47 s


**Fill this in:**

| Shuffle partitions | Wall seconds | Tasks in stage 2 | Notes |
|---|---|---|---|
| 200 | | | |
| 100 | | | |
| 64 | | | |
| 32 | | | |
| 16 | | | |
| 8 | | | |
| 4 | | | |
| 2 | | | |

- Floor was at n = , at seconds
- The curve gets worse below that because:
- The curve gets worse above that because:

In [ ]:
# Now turn AQE on and run the same aggregation once more, at the default 200.
#   spark.conf.set("spark.sql.adaptive.enabled", "true")
#
# Compare it to your 200 row and to your floor. AQE coalesces shuffle partitions
# at runtime, so the question to answer is: did it find your floor on its own?
#
# Turn it back off afterwards, the rest of the notebook assumes off.

**AQE on, 200 partitions:** ______ seconds. Compared to my hand-tuned floor of ______ seconds.

What that tells me about when hand-tuning is still worth doing:

---
## E3 · `repartition` versus `coalesce`

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Shuffles? | **Yes**, full | **No**, merges neighbours |
| Can increase count? | Yes | **No**, decrease only |
| Balance | Even | Can be badly uneven |
| By column? | Yes, `repartition("col")` | No |

In [ ]:
# Build both: df.repartition(16) and df.coalesce(16).
# Print the partition count of each. One of them will surprise you.

In [ ]:
# Time the same aggregation on each, and check the stage count in the UI.
# repartition adds a stage. coalesce does not. Confirm that rather than assume it.

In [ ]:
# Inspect the balance. spark_partition_id() from pyspark.sql.functions tags each
# row with the partition it lives in, so grouping by it gives rows per partition.
#
# Do this for both, and compare the spread between the largest and smallest.
# This is the difference the table above calls "balance", made into a number.

**Fill this in:**

| | `repartition(16)` | `coalesce(16)` |
|---|---|---|
| Actual partitions | | |
| Extra stage? | | |
| Aggregation seconds | | |
| Rows in largest partition | | |
| Rows in smallest partition | | |

**The `coalesce(1)` trap**, in my own words:

---
## E4 · The small files problem

Writing with 200 shuffle partitions produces 200 files. On 153 MB those are a few hundred KB each.

It hurts because every downstream read pays a per-file open cost, object stores bill per request,
and metadata operations get slow. This is the most common real-world Spark output bug.

In [ ]:
# Write the same DataFrame twice, into data/out/default/ and data/out/coalesced/.
# Leave one at whatever partitioning it already has, coalesce the other to 8.
# Use mode("overwrite") so the cell is re-runnable.
#
# data/ is gitignored, so nothing here will be committed.

In [ ]:
# Count the files in each directory and report total size and average file size.
# Ignore the _SUCCESS marker and any .crc files in the count.

In [ ]:
# Time reading each directory back and counting the rows. Same data, same row
# count, different file layout. The gap is what small files cost you.

**Fill this in:**

| | default | coalesced(8) |
|---|---|---|
| Files written | | |
| Total size | | |
| Average file size | | |
| Read-back seconds | | |

---
## E5 · Partition pruning

`partitionBy` on write creates a directory per value: `out/y=2024/m=03/`. A reader filtering on
`m` then skips the other directories without opening them.

⚠️ **Never `partitionBy` a high-cardinality column.** `partitionBy("PULocationID")` would give 265
values times months of directories, and you would have rebuilt the small files problem while
believing you optimised.

In [ ]:
# Derive year and month columns from tpep_pickup_datetime, then write the data
# partitioned by both into data/out/parted/.

In [ ]:
# Look at the directory structure that produced, a couple of levels deep.

In [ ]:
# Read it back, filter to a single month, and explain("formatted").
#
# Find PartitionFilters in the scan node. Compare "number of files read" against
# the unfiltered read. That difference is pruning, and it happened before any
# row was decoded.

**Fill this in:**

- `PartitionFilters` line from the plan:
- Files read with the filter vs without:
- How this differs from `PushedFilters`, which I saw on Day 1:

---
## E6 · Force a spill

**Seeing a spill once means recognising it forever.** Spill means a partition did not fit in
memory and went to disk, and it almost always means partitions are too large.

⚠️ **The obvious version of this exercise does not work, and it is worth knowing why.** Setting
shuffle partitions to 1 and running `groupBy("PULocationID").count()` will **not** spill, because
map-side combine reduces each partition to ~262 rows before anything moves. Only about 786 rows
cross the shuffle. There is nothing to spill.

To spill you need an operation that moves **real rows**, not pre-aggregated summaries. A full sort
does exactly that: every row has to be range-partitioned and sorted.

In [ ]:
# Set shuffle partitions to 1, then force a full sort of the whole dataset and
# trigger it with an action. Select a handful of columns rather than all 19,
# so that this spills rather than simply dying.
#
# Then open the stage detail in the UI and find "Spill (Memory)" and
# "Spill (Disk)" in the Summary Metrics table.
#
# If this OOMs instead of spilling, drop to a single month or fewer columns.
# On 2 cores with a 4 GB driver the margin is not wide.

**Fill this in:**

- Spill (Memory):
- Spill (Disk):
- Wall time:
- Same sort at a sensible partition count, for comparison:

---
## Findings

> Copy into `knowledge_hub/skills/spark/results.md` at the end of the day.

### Measured

| | |
|---|---|
| Input partitions, and why | |
| ⭐ Shuffle partitions at the floor | |
| Runtime at 200 vs at the floor | |
| AQE on, at 200 | |
| `repartition(16)` vs `coalesce(16)` runtime | |
| Partition balance, largest vs smallest | |
| Files written, default vs coalesced | |
| Read-back time, default vs coalesced | |
| Spill (Disk) at 1 shuffle partition | |

### Open questions carried into Day 3

-

### Can I say these out loud, unaided?

Answer out loud **before** reading the block under each. Fill the `___` slots with your own
measurements, because the number is what makes the answer yours.

---

**1. Why is `spark.sql.shuffle.partitions = 200` usually wrong?**  `[ ]`

> Because it is a fixed default that **does not scale with the data**. On a few GB it gives tiny
> partitions and you pay task setup 200 times for almost no work each. On a few TB it gives
> partitions too large for memory, so they spill. The target is roughly **100 to 200 MB per shuffle
> partition**, and at least as many partitions as you have cores.
>
> I benchmarked 200 down to 2 on 9.5M rows and the floor was **___ partitions at ___ seconds**,
> against **___ seconds** at the default. On my data the shuffle itself was tiny because of map-side
> combine, so what I was really measuring was scheduling overhead, and that is still the point:
> 200 tasks that each do nothing still cost 200 task setups.

---

**2. `repartition` or `coalesce`, and when is each wrong?**  `[ ]`

> `coalesce` avoids a shuffle by merging neighbouring partitions, but it can only reduce, and it can
> leave partitions badly uneven. `repartition` costs a full shuffle and gives you even balance, or a
> key-based layout with `repartition("col")`.
>
> I use `coalesce` to cut file count just before a write, and `repartition` when balance or key
> locality matters. In my run `coalesce(16)` left ___ rows in the largest partition against ___ in
> the smallest, while `repartition(16)` was even.

---

**3. What is the `coalesce(1)` trap?**  `[ ]`

> It looks like a free way to get one output file, since it does not shuffle. But because it does
> not shuffle, it **collapses the parallelism of everything upstream too**: the whole computation
> can end up running on one core. If I genuinely need one file, `repartition(1)` is often faster
> despite paying for the shuffle, because the work before it stays parallel.

---

**4. You have 10,000 tiny output files. What happened?**  `[ ]`

> The write inherited the shuffle partition count, and one file is produced per partition. With the
> default 200, or a repartition to something large, on a small dataset you get many tiny files.
>
> It matters because every downstream read pays a per-file open cost, object stores charge per
> request, and metadata operations slow down. Fix: `coalesce` before writing, set shuffle partitions
> to match the data, or let AQE coalesce them. In my run, default gave ___ files averaging ___ and
> read back in ___ seconds, against ___ files and ___ seconds after coalescing.

---

**5. How do you know a partition spilled, and what do you do about it?**  `[ ]`

> Stage detail in the Spark UI, Summary Metrics, the **Spill (Memory)** and **Spill (Disk)** rows.
> Spill means a partition did not fit in execution memory and went to disk, so you paid disk I/O
> inside what should have been an in-memory operation.
>
> The fix is almost always **more partitions**, so each holds less, rather than more memory. I forced
> one by sorting the full dataset with `spark.sql.shuffle.partitions = 1` and saw ___ of disk spill.
>
> Worth knowing: a `groupBy().count()` will not spill however few partitions you give it, because
> map-side combine reduces each partition to a handful of rows before anything moves.

---

**6. Three different things share the word "partition". Name them.**  `[ ]`

> **Input partitions**, how the read is split, driven by file size and `maxPartitionBytes`.
> **Shuffle partitions**, how many partitions exist after a wide operation, set by
> `spark.sql.shuffle.partitions`. And **`partitionBy` on write**, which is directory layout on disk
> for pruning, and has nothing to do with either of the other two.